In [1]:
import requests

In [2]:
data_api = "https://data-api.polymarket.com"
gamma_api = "https://gamma-api.polymarket.com"

In [3]:
def get_market_by_slug(slug):
    """Fetch full market dict from gamma API. Returns None if not found."""
    r = requests.get(f"{gamma_api}/markets/slug/{slug}", timeout=10)
    return r.json() if r.status_code == 200 else None

def get_condition_id(slug):
    m = get_market_by_slug(slug)
    return m.get("conditionId") if m else None

In [4]:
def get_trade_user_market(slug, wallet):
    url = f"{data_api}/trades?user={wallet}&market={get_condition_id(slug)}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()

In [5]:
slug = "btc-updown-15m-1773508500"
wallet = "0x63ce342161250d705dc0b16df89036c8e5f9ba9a"

In [6]:
get_condition_id(slug)

'0xf45ee94a03accb9bff9b9af8609fb2946746acc23d51e9771d589b09ebd6c271'

In [7]:
get_trade_user_market(slug, wallet)

[{'proxyWallet': '0x63ce342161250d705dc0b16df89036c8e5f9ba9a',
  'side': 'BUY',
  'asset': '90914039299415839467311253861927649591838329617429425749301056033772644826176',
  'conditionId': '0xf45ee94a03accb9bff9b9af8609fb2946746acc23d51e9771d589b09ebd6c271',
  'size': 5,
  'price': 0.36,
  'timestamp': 1773509007,
  'title': 'Bitcoin Up or Down - March 14, 1:15PM-1:30PM ET',
  'slug': 'btc-updown-15m-1773508500',
  'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png',
  'eventSlug': 'btc-updown-15m-1773508500',
  'outcome': 'Up',
  'outcomeIndex': 0,
  'name': '0x8dxd',
  'pseudonym': 'Blushing-Fine',
  'bio': '',
  'profileImage': '',
  'profileImageOptimized': '',
  'transactionHash': '0xd38a0ab9e2524f1149d8ab253c2ad3abe1db5a10fbda2a10cb448e1a17f77601'},
 {'proxyWallet': '0x63ce342161250d705dc0b16df89036c8e5f9ba9a',
  'side': 'SELL',
  'asset': '90914039299415839467311253861927649591838329617429425749301056033772644826176',
  'conditionId': '0xf45ee94a03acc

In [11]:
# Fix: WebSocketApp comes from websocket-client (not 'websocket'). Run once, then restart kernel:
%pip install websocket-client

Note: you may need to restart the kernel to use updated packages.


In [8]:
import websocket
import json
import threading
import time

In [9]:
CLOB_WS = "wss://ws-subscriptions-clob.polymarket.com/ws/market"
RTDS_WS  = "wss://ws-live-data.polymarket.com"


In [11]:
def stream_btc_price(on_price):
    """Stream live BTC price from Polymarket RTDS. Calls on_price(value) on each update."""
    def on_open(ws):
        ws.send(json.dumps({"action": "subscribe", "subscriptions": [{"topic": "crypto_prices_chainlink", "type": "*", "filters": '{"symbol":"btc/usd"}'}]}))
        def ping():
            while True:
                time.sleep(5)
                try: ws.send("PING")
                except Exception: break
        threading.Thread(target=ping, daemon=True).start()

    def on_message(ws, message):
        try:
            p = json.loads(message).get("payload", {})
            if p.get("symbol") == "btc/usd":
                on_price(p["value"])
        except (json.JSONDecodeError, KeyError):
            pass

    websocket.WebSocketApp(RTDS_WS,
        on_open=on_open, on_message=on_message,
        on_error=lambda ws, e: None, on_close=lambda ws, s, m: None,
    ).run_forever()


In [12]:
import re

VARIANT_BY_INTERVAL = {5: "fiveminute", 15: "fifteen"}


def _ws_ping(ws, interval=10):
    """Send text PING on ws every interval seconds (daemon thread)."""
    def _loop():
        while True:
            time.sleep(interval)
            try: ws.send("PING")
            except Exception: break
    threading.Thread(target=_loop, daemon=True).start()


def get_price_to_beat(slug):
    """Return PTB for slug from previous window's closePrice."""
    m = re.search(r"btc-updown-(\d+)m-", slug)
    if not m: return None
    variant = VARIANT_BY_INTERVAL.get(int(m.group(1)))
    if not variant: return None
    market = get_market_by_slug(slug)
    if not market: return None
    start_iso = (market.get("events") or [{}])[0].get("startTime", "")
    if not start_iso: return None
    r = requests.get(
        "https://polymarket.com/api/past-results",
        params={"symbol": "BTC", "variant": variant, "assetType": "crypto",
                "currentEventStartTime": start_iso, "count": 1},
        timeout=10,
    )
    if r.status_code != 200: return None
    results = r.json().get("data", {}).get("results", [])
    return results[-1].get("closePrice") if results else None


def get_active_market(interval_min=5):
    """Return info dict for the current open BTC Up/Down market."""
    from datetime import datetime
    now, step = int(time.time()), interval_min * 60
    base = (now // step) * step
    for ts in [base, base + step, base + 2 * step]:
        slug = f"btc-updown-{interval_min}m-{ts}"
        m = get_market_by_slug(slug)
        if m and not m.get("closed", True):
            start_iso = (m.get("events") or [{}])[0].get("startTime", "")
            token_ids = json.loads(m["clobTokenIds"])
            outcomes  = json.loads(m["outcomes"])
            return {
                "slug": slug,
                "question": m["question"],
                "price_to_beat": get_price_to_beat(slug),
                "token_map": dict(zip(outcomes, token_ids)),
            }
    return None


state = {"ptb": None, "btc": None, "up_bid": None, "up_ask": None, "dn_bid": None, "dn_ask": None}

def _print_state():
    ptb = f"${state['ptb']:,.2f}" if state["ptb"] is not None else "..."
    btc = f"${state['btc']:,.2f}" if state["btc"] is not None else "..."
    up  = f"bid={state['up_bid']:.3f} ask={state['up_ask']:.3f}" if state["up_bid"] is not None else "..."
    dn  = f"bid={state['dn_bid']:.3f} ask={state['dn_ask']:.3f}" if state["dn_bid"] is not None else "..."
    print(f"  PTB: {ptb:>12}  BTC: {btc:>12}  |  Up {up}  Down {dn}", flush=True)

In [13]:
def stream_market_price(on_update, interval_min=5):
    """Stream best bid/ask for Up and Down outcomes of the current active market.

    Calls on_update(outcome, bid, ask) on each best_bid_ask tick.
    outcome is 'Up' or 'Down'.
    """
    market = get_active_market(interval_min)
    if not market:
        raise RuntimeError(f"No active {interval_min}m market found")

    token_map = market["token_map"]  # {"Up": token_id, "Down": token_id}
    id_to_outcome = {v: k for k, v in token_map.items()}

    def on_open(ws):
        ws.send(json.dumps({
            "assets_ids": list(token_map.values()),
            "type": "market",
            "custom_feature_enabled": True,
        }))
        _ws_ping(ws, interval=10)

    def on_message(ws, message):
        if message == "PONG": return
        try:
            events = json.loads(message)
            if not isinstance(events, list): events = [events]
            for ev in events:
                if ev.get("event_type") != "best_bid_ask": continue
                outcome = id_to_outcome.get(ev["asset_id"])
                if not outcome: continue
                on_update(outcome, float(ev.get("best_bid", 0)), float(ev.get("best_ask", 0)))
        except (json.JSONDecodeError, ValueError, KeyError):
            pass

    websocket.WebSocketApp(CLOB_WS,
        on_open=on_open, on_message=on_message,
        on_error=lambda ws, e: None, on_close=lambda ws, s, m: None,
    ).run_forever()


In [14]:
market = get_active_market(interval_min=15)  # change to 15 for 15-min markets
print(market['price_to_beat'])

# stream_btc_price(print)
stream_market_price(lambda outcome, bid, ask: print(f"{outcome}: bid={bid:.3f} ask={ask:.3f}"), interval_min=5)

68896.81477
Up: bid=0.480 ask=0.490
Down: bid=0.510 ask=0.520
Up: bid=0.470 ask=0.490
Down: bid=0.510 ask=0.530
Up: bid=0.470 ask=0.490
Down: bid=0.510 ask=0.530
Up: bid=0.460 ask=0.490
Down: bid=0.510 ask=0.540
Up: bid=0.460 ask=0.490
Down: bid=0.510 ask=0.540
Up: bid=0.460 ask=0.490
Down: bid=0.510 ask=0.540
Up: bid=0.460 ask=0.490
Down: bid=0.510 ask=0.540
Up: bid=0.440 ask=0.490
Down: bid=0.510 ask=0.560
Up: bid=0.440 ask=0.490
Down: bid=0.510 ask=0.560
Up: bid=0.440 ask=0.490
Down: bid=0.510 ask=0.560
Up: bid=0.430 ask=0.490
Down: bid=0.510 ask=0.570
Up: bid=0.430 ask=0.490
Down: bid=0.510 ask=0.570
Up: bid=0.430 ask=0.490
Down: bid=0.510 ask=0.570
Up: bid=0.430 ask=0.490
Down: bid=0.510 ask=0.570
Up: bid=0.430 ask=0.490
Down: bid=0.510 ask=0.570
Up: bid=0.430 ask=0.490
Down: bid=0.510 ask=0.570
Up: bid=0.430 ask=0.490
Down: bid=0.510 ask=0.570
Up: bid=0.430 ask=0.490
Down: bid=0.510 ask=0.570
Up: bid=0.430 ask=0.490
Down: bid=0.510 ask=0.570
Up: bid=0.430 ask=0.490
Down: bid=0.51